# PCA — zadanie (Wine)


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import load_wine
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA


In [ ]:
wine = load_wine()
X = pd.DataFrame(wine.data, columns=wine.feature_names)
y = pd.Series(wine.target, name="target")
target_names = wine.target_names
X.head(), y.value_counts().sort_index()


## Standaryzacja

In [ ]:
scaler = StandardScaler()
X_std = scaler.fit_transform(X)


## PCA na wszystkich składowych

In [ ]:
pca = PCA()
X_pca = pca.fit_transform(X_std)

evr = pd.Series(pca.explained_variance_ratio_, name="explained_variance_ratio")
cum = evr.cumsum().rename("cumulative")
pd.concat([evr, cum], axis=1).head(10)


In [ ]:
k80 = int(np.argmax(cum.values >= 0.80) + 1)
k80


## Wykres (scree + skumulowana wariancja)

In [ ]:
plt.figure()
plt.plot(np.arange(1, len(evr)+1), evr.values, marker="o", label="EVR")
plt.plot(np.arange(1, len(cum)+1), cum.values, marker="o", label="Cumulative")
plt.axhline(0.8, linestyle="--")
plt.axvline(k80, linestyle="--")
plt.xlabel("Numer składowej")
plt.ylabel("Udział wariancji")
plt.title("PCA — explained variance ratio")
plt.legend()
plt.tight_layout()
plt.show()


## Projekcja na PC1 i PC2

In [ ]:
pca2 = PCA(n_components=2)
X2 = pca2.fit_transform(X_std)

df2 = pd.DataFrame(X2, columns=["PC1","PC2"])
df2["class"] = [target_names[i] for i in y]
df2.head()


In [ ]:
plt.figure()
for cls in df2["class"].unique():
    sub = df2[df2["class"] == cls]
    plt.scatter(sub["PC1"], sub["PC2"], label=cls)
plt.xlabel("PC1")
plt.ylabel("PC2")
plt.title("Wine — PCA (PC1 vs PC2)")
plt.legend()
plt.tight_layout()
plt.show()


## Loadings (PC1, PC2) + biplot

In [ ]:
loadings = pd.DataFrame(
    pca2.components_.T,
    index=wine.feature_names,
    columns=["PC1","PC2"]
)
loadings.sort_values(by="PC1", key=lambda s: s.abs(), ascending=False).head(10)


In [ ]:
def top_loadings(loadings_df, comp, n=5):
    s = loadings_df[comp].sort_values()
    return pd.concat([s.head(n), s.tail(n)])

top_pc1 = top_loadings(loadings, "PC1", 5)
top_pc2 = top_loadings(loadings, "PC2", 5)
top_pc1, top_pc2


In [ ]:
plt.figure()

for cls in df2["class"].unique():
    sub = df2[df2["class"] == cls]
    plt.scatter(sub["PC1"], sub["PC2"], label=cls, alpha=0.8)

scale = 3.0
for feat in loadings.index:
    x, yv = loadings.loc[feat, "PC1"], loadings.loc[feat, "PC2"]
    plt.arrow(0, 0, x*scale, yv*scale, head_width=0.05, length_includes_head=True)
    plt.text(x*scale*1.05, yv*scale*1.05, feat, fontsize=8)

plt.xlabel("PC1")
plt.ylabel("PC2")
plt.title("Biplot (PC1, PC2)")
plt.legend()
plt.tight_layout()
plt.show()


## PCA bez standaryzacji (porównanie)

In [ ]:
pca_ns = PCA()
X_pca_ns = pca_ns.fit_transform(X.values)

evr_ns = pd.Series(pca_ns.explained_variance_ratio_, name="evr_no_scaling")
cum_ns = evr_ns.cumsum().rename("cum_no_scaling")
pd.concat([evr_ns, cum_ns], axis=1).head(10)


In [ ]:
plt.figure()
plt.plot(np.arange(1, len(evr_ns)+1), evr_ns.values, marker="o", label="EVR (no scaling)")
plt.plot(np.arange(1, len(cum_ns)+1), cum_ns.values, marker="o", label="Cumulative (no scaling)")
plt.axhline(0.8, linestyle="--")
plt.xlabel("Numer składowej")
plt.ylabel("Udział wariancji")
plt.title("PCA bez standaryzacji — explained variance ratio")
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:
pca2_ns = PCA(n_components=2)
X2_ns = pca2_ns.fit_transform(X.values)

df2_ns = pd.DataFrame(X2_ns, columns=["PC1","PC2"])
df2_ns["class"] = [target_names[i] for i in y]

plt.figure()
for cls in df2_ns["class"].unique():
    sub = df2_ns[df2_ns["class"] == cls]
    plt.scatter(sub["PC1"], sub["PC2"], label=cls, alpha=0.8)
plt.xlabel("PC1")
plt.ylabel("PC2")
plt.title("Wine — PCA bez standaryzacji (PC1 vs PC2)")
plt.legend()
plt.tight_layout()
plt.show()
